Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTETomek

In [2]:
df=pd.read_csv('/content/tel_churn.csv')

In [3]:
df.head()

,SeniorCitizen,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_1 - 12,tenure_group_13 - 24,tenure_group_25 - 36,tenure_group_37 - 48,tenure_group_49 - 60,tenure_group_61 - 72
0,0,29.85,29.85,0,True,False,False,True,True,False,...,False,False,True,False,True,False,False,False,False,False
1,0,56.95,1889.50,0,False,True,True,False,True,False,...,False,False,False,True,False,False,True,False,False,False
2,0,53.85,108.15,1,False,True,True,False,True,False,...,False,False,False,True,True,False,False,False,False,False
3,0,42.30,1840.75,0,False,True,True,False,True,False,...,True,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,1,True,False,True,False,True,False,...,False,False,True,False,True,False,False,False,False,False


In [4]:
x=df.drop('Churn',axis=1)
y=df['Churn']

Splitting the data

In [5]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [6]:
print("X Train Length:", len(x_train))
print("X Test Length:", len(x_test))
print("Y Train Length:", len(y_train))
print("Y Test Length:", len(y_test))

X Train Length: 5625
X Test Length: 1407
Y Train Length: 5625
Y Test Length: 1407


In [7]:
print("X Train Shape:", x_train.shape)
print("X Test Shape:", x_test.shape)
print("Y Train Shape:", y_train.shape)
print("Y Test Shape:", y_test.shape)

X Train Shape: (5625, 50)
X Test Shape: (1407, 50)
Y Train Shape: (5625,)
Y Test Shape: (1407,)


DecisionTree Classifier

In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

model_dt = DecisionTreeClassifier(criterion="gini",random_state=100,max_depth=6,min_samples_leaf=8)

model_dt.fit(x_train, y_train)

y_pred = model_dt.predict(x_test)

print("Accuracy :", model_dt.score(x_test, y_test))
print(classification_report(y_test, y_pred))

Accuracy : 0.7668798862828714
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      1033
           1       0.56      0.59      0.57       374

    accuracy                           0.77      1407
   macro avg       0.70      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407



*  F1 score is very small,which shows that the model is weak
*  Precision = 56%\
Recall    = 59%\
F1        = 57%
*  The model is not able to identify the Churn customers

RandomForest Classifier

In [10]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    random_state=100,
    max_depth=6,
    min_samples_leaf=8
)

model_rf.fit(x_train, y_train)

y_pred_rf = model_rf.predict(x_test)

print("Accuracy :", model_rf.score(x_test, y_test))
print(classification_report(y_test, y_pred_rf))

Accuracy : 0.7789623312011372
              precision    recall  f1-score   support

           0       0.81      0.91      0.86      1033
           1       0.63      0.42      0.50       374

    accuracy                           0.78      1407
   macro avg       0.72      0.66      0.68      1407
weighted avg       0.76      0.78      0.76      1407



 Decision Tree Accuracy: 76.69%\
 Random Forest Accuracy: 77.90%\
 Random Forest gives better accuracy than Decision Tree.\
 But it misses many churn customers because the dataset is imbalanced.\
Hence we can apply SMOTEENN to balance the dataset and improve churn prediction.

In [11]:
from imblearn.combine import SMOTEENN

sm = SMOTEENN(random_state=42)

X_resampled, y_resampled = sm.fit_resample(x, y)

In [12]:
xr_train, xr_test, yr_train, yr_test = train_test_split(X_resampled,y_resampled,test_size=0.2,random_state=42)

In [13]:
model_rf_smote = RandomForestClassifier(n_estimators=100,criterion='gini',random_state=100,max_depth=6,min_samples_leaf=8)
model_rf_smote.fit(xr_train, yr_train)
yr_predict = model_rf_smote.predict(xr_test)
print("Accuracy :", model_rf_smote.score(xr_test, yr_test))
print(classification_report(yr_test, yr_predict))

Accuracy : 0.9390862944162437
              precision    recall  f1-score   support

           0       0.96      0.91      0.93       544
           1       0.92      0.97      0.94       638

    accuracy                           0.94      1182
   macro avg       0.94      0.94      0.94      1182
weighted avg       0.94      0.94      0.94      1182



SMOTEENN balanced the dataset and improved churn prediction.\
Hence,This model was selected as the final model.

In [15]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(yr_test, yr_predict)
print(cm)

[[494  50]
 [ 22 616]]


In [16]:
import pickle

pickle.dump(model_rf_smote, open('model.sav', 'wb'))

print("Model Saved")

Model Saved


In [17]:
loaded_model = pickle.load(open('model.sav', 'rb'))

score = loaded_model.score(xr_test, yr_test)

print("Loaded Model Accuracy :", score)

Loaded Model Accuracy : 0.9390862944162437


Our final model i.e. RF Classifier with SMOTEENN, is now ready and dumped in model.sav,\
which we will use and prepare API's so that we can access our model from UI.
